# 02 · 최종 test 평가 (설정 확정 후 1회)

이 노트북은 체크포인트와 평가 규칙을 확정한 뒤에만 실행합니다. Test 결과를 보고 설정을 바꾸거나 반복 선택하면 test 누수가 됩니다. 기본 체크포인트 경로를 확인하고 실행하세요.

In [ ]:
from pathlib import Path
import sys, shutil
import torch
from torch.utils.data import DataLoader
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT_OVERRIDE = None  # 필요하면 프로젝트 경로 지정
if PROJECT_ROOT_OVERRIDE: PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
DATA_DIR = PROJECT_ROOT / 'Data'
if IN_COLAB:
    ACTIVE_DATA_DIR = Path('/content/cost2100_indoor'); ACTIVE_DATA_DIR.mkdir(exist_ok=True)
    for name in ['DATA_Htestin.mat']:
        src, dst = DATA_DIR / name, ACTIVE_DATA_DIR / name
        assert src.is_file(), f'필수 데이터 파일 없음: {src}'
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size: shutil.copy2(src, dst)
else: ACTIVE_DATA_DIR = DATA_DIR
sys.path.insert(0, str(PROJECT_ROOT))
from src.cost2100 import COST2100Dataset
from src.models.csinet import CsiNet
from src.metrics import sample_nmse_linear
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = PROJECT_ROOT / 'runs' / 'csinet_indoor_cr4' / 'best.pt'
print('Checkpoint:', CHECKPOINT, '| exists:', CHECKPOINT.is_file())

In [ ]:
assert CHECKPOINT.is_file(), '체크포인트가 없습니다. 먼저 기준선 학습을 완료하세요.'
test_ds = COST2100Dataset(ACTIVE_DATA_DIR / 'DATA_Htestin.mat')
test_loader = DataLoader(test_ds, batch_size=200, shuffle=False, num_workers=0)
model = CsiNet(encoded_dim=512).to(DEVICE)
state = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
model.load_state_dict(state['model']); model.eval()
ratios = []
with torch.no_grad():
    for x in test_loader:
        x = x.to(DEVICE)
        ratios.append(sample_nmse_linear(x, model(x)).cpu())
ratios = torch.cat(ratios)
test_nmse_db = 10 * torch.log10(ratios.mean().clamp_min(1e-12)).item()
print(f'Test samples: {len(test_ds)} | sample-mean NMSE: {test_nmse_db:.4f} dB')
out = PROJECT_ROOT / 'runs' / 'csinet_indoor_cr4' / 'final_test_sample_nmse.pt'
torch.save({'sample_nmse_linear': ratios, 'mean_nmse_db': test_nmse_db}, out)
print('Saved:', out)